## 初始化库，加载分层后的训练集😆

In [22]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

plt.style.use("default")  # 先加载默认样式
# 设置中文显示+图表样式
plt.rcParams["font.family"] = "Microsoft YaHei"
plt.rcParams["axes.unicode_minus"] = False

print("✅ 依赖库导入完成！")
# 训练集路径（需与之前「分层抽样」Notebook的保存路径一致）
train_path = "../data/processed/train_set.csv"

# 检查文件是否存在，避免路径错误
if not os.path.exists(train_path):
    print(f"❌ 错误：训练集文件未找到！请先运行「数据清洗+分层抽样」Notebook，生成 {train_path}")
else:
    # 这里不使用 index_col，避免误把第一列业务特征当作索引读掉
    df_train = pd.read_csv(train_path)
    print("=" * 60)
    print("✅ 训练集加载成功！")
    print(f"使用路径：{train_path}")
    print(f"训练集规模：{df_train.shape[0]} 行 × {df_train.shape[1]} 列")
    print(f"训练集列名：{df_train.columns.tolist()}")
    print("目标变量（违约标识）：SeriousDlqin2yrs（1=违约，0=正常）")
    print("=" * 60)

✅ 依赖库导入完成！
✅ 训练集加载成功！
使用路径：../data/processed/train_set.csv
训练集规模：119780 行 × 11 列
训练集列名：['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'MonthlyIncome', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'NumberOfDependents', 'SeriousDlqin2yrs']
目标变量（违约标识）：SeriousDlqin2yrs（1=违约，0=正常）


## 训练集缺失值分析（明确填充对象）

根据EDA可知，月收入和家属收入有数据缺失
- 针对MonthlyIncome（月收入）字段中存在的 0 值
    - 业务层面：信用卡授信要求用户具备稳定还款能力，月收入为 0 的样本不具备真实业务代表性；
    - 数据层面：0 值会使收入分布极端偏斜，会干扰模型对还款能力特征的学习；
    - (💥**new**)处理方案：采用随机森林填充策略

In [23]:
from sklearn.ensemble import RandomForestRegressor


def set_missing(df) -> "pd.DataFrame":
    # 填充目标列
    fill_col = "MonthlyIncome"

    # 参考列（与原思路一致，包含目标变量用于监督学习）
    reference_columns = [
        "RevolvingUtilizationOfUnsecuredLines",
        "age",
        "NumberOfTime30-59DaysPastDueNotWorse",
        "DebtRatio",
        "NumberOfOpenCreditLinesAndLoans",
        "NumberOfTimes90DaysLate",
        "NumberRealEstateLoansOrLines",
        "NumberOfTime60-89DaysPastDueNotWorse",
        "SeriousDlqin2yrs",
    ]

    required_columns = [fill_col] + reference_columns
    missing_columns = [c for c in required_columns if c not in df.columns]
    if missing_columns:
        raise ValueError(f"缺少必要列，无法填充 {fill_col}: {missing_columns}")

    # 1. 选取数值型特征，把待填充的「月收入」放到第一列
    process_df = df[[fill_col] + reference_columns].copy()

    # 2. 拆分数据：已知月收入的行 + 缺失月收入的行
    known = process_df[process_df[fill_col].notnull()].values
    unknown = process_df[process_df[fill_col].isnull()].values

    if unknown.shape[0] == 0:
        print("✅ MonthlyIncome 没有缺失值，无需填充。")
        print(f"填充列名: {fill_col}")
        print(f"使用参考列名: {reference_columns}")
        return df.copy()

    # 3. 划分特征X和标签y（训练模型用）
    X = known[:, 1:]  # 除月收入外的所有列（特征）
    y = known[:, 0]   # 第一列：月收入（预测目标）

    # 4. 初始化并训练随机森林回归模型
    rfr = RandomForestRegressor(
        random_state=0,
        n_estimators=200,
        max_depth=3,
        n_jobs=-1,
    )
    rfr.fit(X, y)

    # 5. 用训练好的模型预测缺失的月收入
    predicted = rfr.predict(unknown[:, 1:]).round(0)

    # 6. 将预测值回填到原数据的缺失位置
    filled_df = df.copy()
    filled_df.loc[filled_df[fill_col].isnull(), fill_col] = predicted

    # 诊断输出
    print(f"填充列名: {fill_col}")
    print(f"使用参考列名: {reference_columns}")

    return filled_df


# 输入：data/processed/train_set.csv（由上一个单元加载到 df_train）
# 输出：
# 1) 变量 df_train_filled（供下一个 cell 继续处理）
# 2) 临时文件 data/temporary/training_set_processing_1.csv（后续统一再导出正式文件）
if "df_train" not in globals():
    raise ValueError("未找到 df_train，请先执行第2个代码单元加载训练集。")

before_missing = int(df_train["MonthlyIncome"].isnull().sum())
df_train_filled = set_missing(df_train)
after_missing = int(df_train_filled["MonthlyIncome"].isnull().sum())

# 兼容 notebook 不同工作目录：优先 data/temporary，其次 ../data/temporary
temp_dir_candidates = [
    os.path.join("data", "temporary"),
    os.path.join("..", "data", "temporary"),
]

temp_dir = temp_dir_candidates[0]
for candidate in temp_dir_candidates:
    parent_dir = os.path.dirname(candidate)
    if os.path.exists(parent_dir):
        temp_dir = candidate
        break

os.makedirs(temp_dir, exist_ok=True)
temp_output_path = os.path.join(temp_dir, "training_set_processing_1.csv")
df_train_filled.to_csv(temp_output_path, index=False)

print("=" * 60)
print("✅ 随机森林填充完成")
print(f"填充前 MonthlyIncome 缺失数: {before_missing}")
print(f"填充后 MonthlyIncome 缺失数: {after_missing}")
print("输出变量: df_train_filled")
print(f"临时文件已保存: {temp_output_path}")
print("=" * 60)

df_train_filled[["MonthlyIncome"]].describe()

填充列名: MonthlyIncome
使用参考列名: ['RevolvingUtilizationOfUnsecuredLines', 'age', 'NumberOfTime30-59DaysPastDueNotWorse', 'DebtRatio', 'NumberOfOpenCreditLinesAndLoans', 'NumberOfTimes90DaysLate', 'NumberRealEstateLoansOrLines', 'NumberOfTime60-89DaysPastDueNotWorse', 'SeriousDlqin2yrs']
✅ 随机森林填充完成
填充前 MonthlyIncome 缺失数: 23811
填充后 MonthlyIncome 缺失数: 0
输出变量: df_train_filled
临时文件已保存: ..\data\temporary\training_set_processing_1.csv


,MonthlyIncome
count,119780.000000
mean,5802.266355
std,7117.827050
min,0.000000
25%,2400.000000
50%,5000.000000
75%,7706.000000
max,730483.000000


- 针对 NumberOfDependents（家属数量）
    - 出现一个断层的20家属，13个家属：同样极不合理，并且都只有1个，建议处理。
    - (💦**new**)NA值：缺失值占比约（3924/总样本数），考虑删除。

- 针对 NumberRealEstateLoansOrLines（房产数量）
    - 出现54的过大值，可能删除

In [24]:
# 针对 NumberOfDependents 缺失值：先统计，再删除缺失行
missing_col = "NumberOfDependents"

if "df_train_filled" not in globals():
    raise ValueError("未找到 df_train_filled，请先执行第4个代码单元。")

if missing_col not in df_train_filled.columns:
    raise ValueError(f"当前数据中不存在列: {missing_col}")

rows_before = len(df_train_filled)
missing_count = int(df_train_filled[missing_col].isnull().sum())
missing_ratio = missing_count / rows_before if rows_before > 0 else 0.0

print("=" * 60)
print(f"🔎 {missing_col} 缺失值检查")
print(f"删除前总行数: {rows_before}")
print(f"缺失行数: {missing_count}")
print(f"缺失占比: {missing_ratio:.2%}")

# 删除 NumberOfDependents 缺失行
df_train_no_dep_missing = df_train_filled[df_train_filled[missing_col].notnull()].copy()

rows_after = len(df_train_no_dep_missing)
deleted_rows = rows_before - rows_after
remaining_missing = int(df_train_no_dep_missing[missing_col].isnull().sum())

print("\n🧹 删除完成")
print(f"删除行数: {deleted_rows}")
print(f"删除后总行数: {rows_after}")
print(f"删除后缺失行数: {remaining_missing}")

# 临时保存：training_set_processing_2.csv（变量继续保留）
temp_dir_candidates = [
    os.path.join("data", "temporary"),
    os.path.join("..", "data", "temporary"),
]

temp_dir = temp_dir_candidates[0]
for candidate in temp_dir_candidates:
    parent_dir = os.path.dirname(candidate)
    if os.path.exists(parent_dir):
        temp_dir = candidate
        break

os.makedirs(temp_dir, exist_ok=True)
temp_output_path_2 = os.path.join(temp_dir, "training_set_processing_2.csv")

try:
    df_train_no_dep_missing.to_csv(temp_output_path_2, index=False)
    print(f"临时文件已保存: {temp_output_path_2}")
except PermissionError:
    fallback_path = os.path.join(temp_dir, "training_set_processing_2_autosave.csv")
    df_train_no_dep_missing.to_csv(fallback_path, index=False)
    print(f"⚠️ 原文件被占用，已改存为: {fallback_path}")

print("输出变量: df_train_no_dep_missing")
print("=" * 60)

df_train_no_dep_missing[[missing_col]].describe()

🔎 NumberOfDependents 缺失值检查
删除前总行数: 119780
缺失行数: 3127
缺失占比: 2.61%

🧹 删除完成
删除行数: 3127
删除后总行数: 116653
删除后缺失行数: 0
临时文件已保存: ..\data\temporary\training_set_processing_2.csv
输出变量: df_train_no_dep_missing


,NumberOfDependents
count,116653.000000
mean,0.759089
std,1.116483
min,0.000000
25%,0.000000
50%,0.000000
75%,1.000000
max,20.000000


### 重复值检查

In [25]:
# 基于当前处理后的变量继续（优先用最新变量）
if "df_train_no_dep_missing" in globals():
    data = df_train_no_dep_missing.copy()
elif "df_train_filled" in globals():
    data = df_train_filled.copy()
else:
    data = df_train.copy()

rows_before = len(data)
dup_rows = int(data.duplicated().sum())  # 全列完全重复的行数

print("=" * 60)
print(f"删除前总行数: {rows_before}")
print(f"检测到重复行数: {dup_rows}")

# 删除重复行
data = data.drop_duplicates()

rows_after = len(data)
print(f"删除后总行数: {rows_after}")
print(f"实际删除行数: {rows_before - rows_after}")
print("=" * 60)

# 继续保留变量 + 临时保存
df_train_no_dup = data.copy()
df_train_no_dup.to_csv("../data/temporary/training_set_processing_3.csv", index=False)

# 显式衔接给下一段异常值处理
df_for_outlier = df_train_no_dup.copy()

print("输出变量: df_train_no_dup")
print("输出变量: df_for_outlier")
print("临时文件: ../data/temporary/training_set_processing_3.csv")

删除前总行数: 116653
检测到重复行数: 347
删除后总行数: 116306
实际删除行数: 347
输出变量: df_train_no_dup
输出变量: df_for_outlier
临时文件: ../data/temporary/training_set_processing_3.csv


### 异常值处理

In [26]:
from pathlib import Path


def outlier_processing(df, col):
    """离群值处理（IQR 规则）"""
    s = df[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return df[(df[col] >= lower) & (df[col] <= upper)]


def outlier_stats(df, col):
    """只做离群检测，不删数据：返回阈值和离群数量"""
    s = df[col]
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr

    mask_outlier = (s < lower) | (s > upper)
    outlier_count = int(mask_outlier.sum())
    total_count = int(s.notna().sum())
    outlier_ratio = (outlier_count / total_count) if total_count else 0.0

    return {
        "column": col,
        "q1": float(q1),
        "q3": float(q3),
        "iqr": float(iqr),
        "lower": float(lower),
        "upper": float(upper),
        "outlier_count": outlier_count,
        "total_count": total_count,
        "outlier_ratio": outlier_ratio,
    }


def resolve_temp_file(candidates):
    for base in [Path("../data/temporary"), Path("data/temporary")]:
        for name in candidates:
            p = base / name
            if p.exists():
                return p
    return None


# 数据源优先级（与上一个单元衔接）
target_col = "SeriousDlqin2yrs"
outlier_base_df = None
outlier_data_source = None

if "df_for_outlier" in globals():
    outlier_base_df = df_for_outlier.copy()
    outlier_data_source = "内存变量: df_for_outlier"
elif "df_train_no_dup" in globals():
    outlier_base_df = df_train_no_dup.copy()
    outlier_data_source = "内存变量: df_train_no_dup"
elif "df_train_no_dep_missing" in globals():
    outlier_base_df = df_train_no_dep_missing.copy()
    outlier_data_source = "内存变量: df_train_no_dep_missing"
elif "df_train_filled" in globals():
    outlier_base_df = df_train_filled.copy()
    outlier_data_source = "内存变量: df_train_filled"
else:
    selected_temp_path = resolve_temp_file(
        [
            "training_set_processing_3.csv",
            "training_set_processing_2.csv",
            "training_set_processing_1.csv",
        ]
    )
    if selected_temp_path is not None:
        outlier_base_df = pd.read_csv(selected_temp_path)
        outlier_data_source = f"temporary 文件: {selected_temp_path}"

if outlier_base_df is None:
    if "df_train" not in globals():
        raise ValueError("未找到可用数据：请先执行前序处理单元或准备 temporary 处理结果文件。")
    outlier_base_df = df_train.copy()
    outlier_data_source = "回退数据: df_train"

numeric_cols = outlier_base_df.select_dtypes(include=[np.number]).columns.tolist()
if target_col in numeric_cols:
    numeric_cols.remove(target_col)

stats_rows = [outlier_stats(outlier_base_df, col) for col in numeric_cols]
outlier_summary_df = pd.DataFrame(stats_rows).sort_values("outlier_ratio", ascending=False)

print("=" * 70)
print("📌 IQR 离群检测结果（按离群占比降序）")
print(f"使用数据源: {outlier_data_source}")
print(f"检测数据规模: {outlier_base_df.shape[0]} 行 × {outlier_base_df.shape[1]} 列")
print(f"检测列数: {len(numeric_cols)}")
print("=" * 70)
print(
    outlier_summary_df[["column", "outlier_count", "total_count", "outlier_ratio", "lower", "upper"]]
    .to_string(index=False, float_format=lambda x: f"{x:.6f}")
)

# 同步保存当前衔接数据，便于下一个处理阶段继续接力
temp_dir = Path("../data/temporary")
temp_dir.mkdir(parents=True, exist_ok=True)
out_path = temp_dir / "training_set_processing_4_outlier_base.csv"
outlier_base_df.to_csv(out_path, index=False)

print("\n输出变量: outlier_summary_df")
print("输出变量: outlier_base_df")
print(f"临时文件: {out_path}")
outlier_summary_df

📌 IQR 离群检测结果（按离群占比降序）
使用数据源: 内存变量: df_for_outlier
检测数据规模: 116306 行 × 11 列
检测列数: 10
                              column  outlier_count  total_count  outlier_ratio        lower        upper
                           DebtRatio          22699       116306       0.195166    -0.723498     1.671553
NumberOfTime30-59DaysPastDueNotWorse          18726       116306       0.161006     0.000000     0.000000
                  NumberOfDependents          10699       116306       0.091990    -1.500000     2.500000
             NumberOfTimes90DaysLate           6342       116306       0.054529     0.000000     0.000000
NumberOfTime60-89DaysPastDueNotWorse           5800       116306       0.049868     0.000000     0.000000
                       MonthlyIncome           3884       116306       0.033395 -5309.000000 15515.000000
     NumberOfOpenCreditLinesAndLoans           3156       116306       0.027135    -4.000000    20.000000
        NumberRealEstateLoansOrLines            628       116306     

,column,q1,q3,iqr,lower,upper,outlier_count,total_count,outlier_ratio
3,DebtRatio,0.174646,0.773409,0.598763,-0.723498,1.671553,22699,116306,0.195166
2,NumberOfTime30-59DaysPastDueNotWorse,0.000000,0.000000,0.000000,0.000000,0.000000,18726,116306,0.161006
9,NumberOfDependents,0.000000,1.000000,1.000000,-1.500000,2.500000,10699,116306,0.091990
6,NumberOfTimes90DaysLate,0.000000,0.000000,0.000000,0.000000,0.000000,6342,116306,0.054529
8,NumberOfTime60-89DaysPastDueNotWorse,0.000000,0.000000,0.000000,0.000000,0.000000,5800,116306,0.049868
4,MonthlyIncome,2500.000000,7706.000000,5206.000000,-5309.000000,15515.000000,3884,116306,0.033395
5,NumberOfOpenCreditLinesAndLoans,5.000000,11.000000,6.000000,-4.000000,20.000000,3156,116306,0.027135
7,NumberRealEstateLoansOrLines,0.000000,2.000000,2.000000,-3.000000,5.000000,628,116306,0.005400
0,RevolvingUtilizationOfUnsecuredLines,0.031005,0.561917,0.530912,-0.765363,1.358285,595,116306,0.005116
1,age,41.000000,62.000000,21.000000,9.500000,93.500000,95,116306,0.000817


### 输出文件

In [27]:
from pathlib import Path

# 优先保存“最新处理结果”
if "outlier_base_df" in globals():
    final_df = outlier_base_df.copy()
    final_source = "outlier_base_df"
elif "df_for_outlier" in globals():
    final_df = df_for_outlier.copy()
    final_source = "df_for_outlier"
elif "df_train_no_dup" in globals():
    final_df = df_train_no_dup.copy()
    final_source = "df_train_no_dup"
elif "df_train_no_dep_missing" in globals():
    final_df = df_train_no_dep_missing.copy()
    final_source = "df_train_no_dep_missing"
elif "df_train_filled" in globals():
    final_df = df_train_filled.copy()
    final_source = "df_train_filled"
elif "df_train" in globals():
    final_df = df_train.copy()
    final_source = "df_train"
else:
    raise ValueError("未找到可保存的数据变量，请先执行前面的处理单元。")

# 保存到 processed 文件夹
processed_dir_candidates = [Path("../data/processed"), Path("data/processed")]
processed_dir = processed_dir_candidates[0]
for p in processed_dir_candidates:
    parent = p.parent
    if parent.exists():
        processed_dir = p
        break

processed_dir.mkdir(parents=True, exist_ok=True)
out_path = processed_dir / "train_set_processed_new.csv"
final_df.to_csv(out_path, index=False)

print("=" * 60)
print("✅ 处理后训练集已保存")
print(f"数据来源变量: {final_source}")
print(f"输出路径: {out_path}")
print(f"数据规模: {final_df.shape[0]} 行 × {final_df.shape[1]} 列")
print("=" * 60)

✅ 处理后训练集已保存
数据来源变量: outlier_base_df
输出路径: ..\data\processed\train_set_processed_new.csv
数据规模: 116306 行 × 11 列
